In [ ]:
!pip install -U transformers>=4.48.0
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.4/487.4 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 18.5 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system 

In [ ]:
from transformers import AutoModel, AutoTokenizer
import os
import sys
import torch
import random
from datasets import load_dataset, Dataset
from tokenizers import BertWordPieceTokenizer
from transformers import ModernBertConfig, ModernBertForMaskedLM, PreTrainedTokenizerFast, DataCollatorForLanguageModeling, Trainer, TrainingArguments


# モデル名
repo_name = "Shuu12121/CodeMorph-ModernBERT"
# Hugging Face からモデルをロード
model = ModernBertForMaskedLM.from_pretrained(repo_name)
tokenizer = AutoTokenizer.from_pretrained(repo_name)

print("モデルのロード成功！")
print(model)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用デバイス: {device}")
model.to(device)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/609M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/408k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.20M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

モデルのロード成功！
ModernBertForMaskedLM(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50000, 768, padding_idx=0)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=6144, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=3072, out_features=768, bias=False)
        )
      )
     

ModernBertForMaskedLM(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50000, 768, padding_idx=0)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=6144, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=3072, out_features=768, bias=False)
        )
      )
      (1-11): 11

In [ ]:
import torch
import numpy as np
import random
import re
import sys
import importlib.util
from datasets import load_dataset
from transformers import AutoModelForMaskedLM, AutoTokenizer, AutoModel, T5ForConditionalGeneration
from sklearn.metrics.pairwise import cosine_similarity

# sentence_transformers がインストールされているか確認
st_available = importlib.util.find_spec("sentence_transformers") is not None
if not st_available:
    print("sentence_transformers がインストールされていません。一部のモデルでエラーが発生する可能性があります。")
    print("インストールするには: pip install sentence-transformers>=2.7.0")
    print("インストールなしで続行します...")
else:
    print("sentence_transformers が利用可能です。")

# ---------------------------------------------------
# ユーティリティ関数群
# ---------------------------------------------------

def remove_multiline_strings(code):
    """
    Pythonコード中のマルチライン文字列（""" """ や ''' '''）を削除します。
    """
    pattern = r'("""(.*?)"""|\'\'\'(.*?)\'\'\')'
    return re.sub(pattern, '', code, flags=re.DOTALL)

def get_cls_embedding(model, tokenizer, text, device, max_length=256):
    """
    入力テキストの埋め込みを取得する関数。
    - BERT/CodeBERT系の場合は CLS トークンの埋め込みを使用。
    - CodeT5系の場合は encoder 出力の平均プーリングを使用。
    - ModernBERT のように内部に model がある場合は model.model を利用。
    - SFR-Embedding-Code モデルには特別な処理を適用。
    """
    model_name = getattr(model, "name_or_path", "")
    is_sfr_model = "SFR-Embedding-Code" in model_name

    # SFR-Embedding-Code モデルの場合、sentence-transformers 互換の呼び出し方法を使用
    if is_sfr_model:
        encoded_input = tokenizer.encode(text, max_length=max_length, truncation=True, return_tensors='pt').to(device)
        with torch.no_grad():
            try:
                # SFR-Embedding-Code モデルでは encode_tokens メソッドを使う
                if hasattr(model, "encode_tokens"):
                    embedding = model.encode_tokens(encoded_input)
                # または forward_features メソッドを使う
                elif hasattr(model, "forward_features"):
                    embedding = model.forward_features(encoded_input)
                # または last_hidden_state を直接取得
                else:
                    outputs = model(encoded_input)
                    embedding = outputs.last_hidden_state[:, 0, :]
                return embedding.detach().cpu().numpy()
            except Exception as e:
                print(f"SFR モデル特有の処理中にエラー発生: {e}")
                # フォールバックとして、encode メソッドを試す
                try:
                    embedding = model.encode([text], convert_to_tensor=True)
                    return embedding.detach().cpu().numpy()
                except Exception as e2:
                    print(f"フォールバック処理中にもエラー発生: {e2}")
                    raise

    # 通常のモデル処理
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
    # ModernBERT の場合、token_type_ids が不要なので削除する
    if "token_type_ids" in inputs:
        inputs.pop("token_type_ids")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        if hasattr(model, "model"):
            outputs = model.model(**inputs)
            embedding = outputs.last_hidden_state[:, 0, :]
        elif hasattr(model, "bert"):
            outputs = model.bert(**inputs)
            embedding = outputs.last_hidden_state[:, 0, :]
        elif hasattr(model, "roberta"):
            outputs = model.roberta(**inputs)
            embedding = outputs.last_hidden_state[:, 0, :]
        elif hasattr(model, "encoder"):
            # CodeT5系の場合は、CLSトークンが存在しないため平均プーリングを使用
            outputs = model.encoder(**inputs)
            embedding = outputs.last_hidden_state.mean(dim=1)
        else:
            try:
                outputs = model(**inputs)
                if hasattr(outputs, "last_hidden_state"):
                    embedding = outputs.last_hidden_state[:, 0, :]
                elif hasattr(outputs, "hidden_states"):
                    embedding = outputs.hidden_states[-1][:, 0, :]
                else:
                    raise ValueError("モデル出力にlast_hidden_stateまたはhidden_statesがありません。")
            except Exception as e:
                print(f"一般的なモデル処理でエラー: {e}")
                # 特殊なモデルの場合は、forward メソッドに直接 input_ids を渡してみる
                try:
                    outputs = model(inputs["input_ids"])
                    embedding = outputs.last_hidden_state[:, 0, :]
                except Exception as e2:
                    print(f"代替処理もエラー: {e2}")
                    raise

    return embedding.detach().cpu().numpy()


def calculate_f1(precision, recall):
    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)

def calculate_dcg(ranking):
    dcg = 0.0
    for i, rel in enumerate(ranking):
        dcg += rel / np.log2(i + 2)
    return dcg

def calculate_metrics(sim_matrix, k_values=[1, 5, 10, 50, 100]):
    """
    候補プール内で正解（常にインデックス0が正解と仮定）に基づいて評価指標を計算します。
    """
    num_queries = sim_matrix.shape[0]
    metrics = {
        "mrr": 0.0,
        "map": 0.0,
        "r_precision": 0.0,
        "recall@k": {k: 0.0 for k in k_values},
        "precision@k": {k: 0.0 for k in k_values},
        "ndcg@k": {k: 0.0 for k in k_values},
        "f1@k": {k: 0.0 for k in k_values},
        "success_rate@k": {k: 0.0 for k in k_values},
        "query_coverage@k": {k: 0.0 for k in k_values},
    }

    for i in range(num_queries):
        sims = sim_matrix[i]
        ranked_indices = np.argsort(-sims)
        # 正解は候補プール内の先頭（インデックス0）
        correct_rank = np.where(ranked_indices == 0)[0][0] + 1
        metrics["mrr"] += 1.0 / correct_rank
        metrics["map"] += 1.0 / correct_rank
        metrics["r_precision"] += 1.0 if correct_rank == 1 else 0.0

        for k in k_values:
            if correct_rank <= k:
                prec = 1.0 / correct_rank
                rec = 1.0
            else:
                prec = 0.0
                rec = 0.0
            f1 = calculate_f1(prec, rec)
            metrics["precision@k"][k] += prec
            metrics["recall@k"][k] += rec
            metrics["f1@k"][k] += f1

            ideal_ranking = [1.0] + [0.0] * (k - 1)
            actual_ranking = [1.0 if j == 0 else 0.0 for j in ranked_indices[:k]]
            idcg = calculate_dcg(ideal_ranking)
            dcg = calculate_dcg(actual_ranking)
            metrics["ndcg@k"][k] += dcg / idcg if idcg > 0 else 0.0

            metrics["success_rate@k"][k] += 1.0 if correct_rank <= k else 0.0
            metrics["query_coverage@k"][k] += 1.0 if correct_rank <= k else 0.0

    metrics["mrr"] /= num_queries
    metrics["map"] /= num_queries
    metrics["r_precision"] /= num_queries
    for k in k_values:
        metrics["precision@k"][k] /= num_queries
        metrics["recall@k"][k] /= num_queries
        metrics["ndcg@k"][k] /= num_queries
        metrics["f1@k"][k] /= num_queries
        metrics["success_rate@k"][k] /= num_queries
        metrics["query_coverage@k"][k] /= num_queries
    return metrics

def display_code_search_results(metrics, model_name):
    """
    評価結果を見やすいテーブル形式で表示します。
    """
    print(f"\n==== {model_name} Code Search Evaluation Results ====")
    print(f"MRR:         {metrics['mrr']:.4f}")
    print(f"MAP:         {metrics['map']:.4f}")
    print(f"R-Precision: {metrics['r_precision']:.4f}\n")

    header = "{:<6}{:<10}{:<10}{:<10}{:<10}{:<15}{:<15}"
    print(header.format("K", "Recall", "Precision", "NDCG", "F1", "Success Rate", "Query Cov."))
    print("-" * 76)
    for k in sorted(metrics["recall@k"].keys()):
        recall = metrics["recall@k"][k]
        precision = metrics["precision@k"][k]
        ndcg = metrics["ndcg@k"][k]
        f1 = metrics["f1@k"][k]
        success_rate = metrics["success_rate@k"][k]
        query_cov = metrics["query_coverage@k"][k]
        print(header.format(k,
                            f"{recall:.4f}",
                            f"{precision:.4f}",
                            f"{ndcg:.4f}",
                            f"{f1:.4f}",
                            f"{success_rate:.4f}",
                            f"{query_cov:.4f}"))

def evaluate_code_search(model, tokenizer, dataset, device, max_examples=100, pool_size=100,
                         query_field="docstring", code_field="code"):
    """
    指定したデータセット上でコード検索タスクを実施し、評価指標を計算します。
    各サンプルに対して、正解は候補プール内の先頭（自身）と仮定しています。
    """
    num_examples = min(len(dataset), max_examples)
    all_codes = [remove_multiline_strings(dataset[i][code_field]) for i in range(num_examples)]
    queries = [dataset[i][query_field] for i in range(num_examples)]

    # 各コードの埋め込み計算
    all_code_embeddings = []
    print("データセット全体のコード埋め込みを計算中...")

    # バッチ処理を実装して効率化
    batch_size = 16
    for i in range(0, num_examples, batch_size):
        batch_codes = all_codes[i:min(i+batch_size, num_examples)]
        batch_embeddings = []
        for code in batch_codes:
            emb = get_cls_embedding(model, tokenizer, code, device)
            batch_embeddings.append(emb)
        all_code_embeddings.extend(batch_embeddings)
    all_code_embeddings = np.vstack(all_code_embeddings)
    print("コード埋め込み計算完了.")

    # 各クエリに対し候補プール内の類似度計算
    sim_matrices = []
    print("候補コードプールを作成し、類似度計算中...")
    for i in range(num_examples):
        query_embedding = get_cls_embedding(model, tokenizer, queries[i], device).reshape(1, -1)
        candidate_indices = [i] + random.sample([j for j in range(num_examples) if j != i], pool_size - 1)
        candidate_embeddings = all_code_embeddings[candidate_indices]
        sims = cosine_similarity(query_embedding, candidate_embeddings)[0]
        sim_matrices.append(sims)

    sim_matrix = np.array(sim_matrices)
    print("類似度計算完了.")

    metrics = calculate_metrics(sim_matrix)
    return metrics

# ---------------------------------------------------
# メイン処理：各モデル・各言語でコード検索実験を実施
# ---------------------------------------------------

if __name__ == "__main__":
    # Python の乱数シードを設定（候補サンプルの抽出などに影響します）
    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"使用デバイス: {device}")

    # ------------------------------
    # ① ModernBERT 単体ロード確認
    # ------------------------------
    repo_name = "Shuu12121/CodeMorph-ModernBERT"
    try:
        model_demo = AutoModelForMaskedLM.from_pretrained(repo_name)
        tokenizer_demo = AutoTokenizer.from_pretrained(repo_name)
        model_demo.to(device)
        print("\n【ModernBERT 単体ロード確認】")
        print("モデルのロード成功！")
        print(model_demo)
    except Exception as e:
        print(f"\n【ModernBERT 単体ロード確認】")
        print(f"モデルのロード失敗: {e}")

    # ------------------------------
    # ② 評価データセットの言語一覧（例）
    # ------------------------------
    languages = ["python", "java", "javascript", "php", "ruby", "go"]
    max_examples = 1000  # メモリと時間の制約のため、各言語ごとに100サンプルに制限

    # ------------------------------
    # ③ 各モデルでコード検索実験を実施
    # ------------------------------
    model_configs = [
        {"name": "microsoft/codebert-base", "class": AutoModelForMaskedLM},
        {"name": "microsoft/graphcodebert-base", "class": AutoModelForMaskedLM},
        {"name": "huggingface/CodeBERTa-small-v1", "class": AutoModelForMaskedLM},
        {"name": "Shuu12121/CodeMorph-ModernBERT-ALT", "class": AutoModelForMaskedLM},
        {"name": "Shuu12121/CodeMorph-ModernBERT", "class": AutoModelForMaskedLM},
        {"name": "Shuu12121/CodeMorph-ModernBERTv2", "class": AutoModelForMaskedLM},
        {"name": "Shuu12121/CodeMorph-ModernBERT-large-vocab", "class": AutoModelForMaskedLM},
        {"name": "Shuu12121/CodeMorph-ModernBERT-Alternative", "class": AutoModelForMaskedLM},
        {"name": "Salesforce/codet5p-220m-bimodal", "class": T5ForConditionalGeneration},
        {"name": "Shuu12121/CodeModernBERT-Snake", "class": AutoModelForMaskedLM, "use_sentence_transformers": False},
        {"name": "Shuu12121/CodeModernBERT-Owl", "class": AutoModelForMaskedLM, "use_sentence_transformers": False},
        {"name": "Salesforce/SFR-Embedding-Code-400M_R", "class": AutoModel, "use_sentence_transformers": True},
        {"name": "Shuu12121/CodeSearch-ModernBERT-Snake", "class": AutoModelForMaskedLM, "use_sentence_transformers": True},
        {"name": "Shuu12121/CodeSearch-ModernBERT-Owl", "class": AutoModelForMaskedLM, "use_sentence_transformers": True},
    ]

    for lang in languages:
        print(f"\n==== 言語: {lang} の評価を開始 ====")
        try:
            # データセットをロード後、シャッフルしてランダムサンプルを抽出
            dataset = load_dataset("google/code_x_glue_ct_code_to_text", lang, split="test", trust_remote_code=True)
            dataset = dataset.shuffle(seed=42)
            subset = dataset.select(range(min(max_examples, len(dataset))))

            for config in model_configs:
                model_name = config["name"]
                model_class = config["class"]
                print(f"\n{model_name} を評価します (候補プールサイズ: 100)...")
                try:
                    use_st = config.get("use_sentence_transformers", False)

                    if use_st:
                        try:
                            # sentence-transformers を使用する場合
                            from sentence_transformers import SentenceTransformer
                            print("SentenceTransformer を使用してモデルをロードします...")
                            model = SentenceTransformer(model_name, trust_remote_code=True)
                            model.to(device)
                            tokenizer = model.tokenizer

                            # get_cls_embedding 関数を使用せず、直接 encode メソッドを使用するための関数を定義
                            def st_get_embedding(text, device):
                                with torch.no_grad():
                                    embeddings = model.encode([text], convert_to_tensor=True)
                                    return embeddings.cpu().numpy()

                            # 元の関数を一時的に保存
                            original_get_cls_embedding = get_cls_embedding
                            # 関数をオーバーライド
                            get_cls_embedding = lambda model, tokenizer, text, device, max_length=256: st_get_embedding(text, device)

                        except (ImportError, Exception) as e:
                            print(f"SentenceTransformer のロードに失敗しました: {e}")
                            print("通常の方法でモデルをロードします...")
                            tokenizer = AutoTokenizer.from_pretrained(model_name)
                            model = model_class.from_pretrained(model_name, trust_remote_code=True)
                    else:
                        # 通常の方法でモデルをロード
                        tokenizer = AutoTokenizer.from_pretrained(model_name)
                        model = model_class.from_pretrained(model_name, trust_remote_code=True)

                    model.to(device)
                    model.eval()  # 評価モードに設定

                    metrics = evaluate_code_search(model, tokenizer, subset, device,
                                               max_examples=len(subset),
                                               pool_size=1000,
                                               query_field="docstring",
                                               code_field="code")
                    display_code_search_results(metrics, f"{model_name} - {lang}")

                    # 元の get_cls_embedding 関数を復元（オーバーライドした場合）
                    if use_st and 'original_get_cls_embedding' in locals():
                        get_cls_embedding = original_get_cls_embedding
                        del original_get_cls_embedding

                    # メモリ解放
                    del model
                    del tokenizer
                    torch.cuda.empty_cache()

                except Exception as e:
                    print(f"{model_name} の評価中にエラーが発生しました: {e}")
        except Exception as e:
            print(f"{lang} のデータセットロード中にエラーが発生しました: {e}")

sentence_transformers が利用可能です。
使用デバイス: cuda

【ModernBERT 単体ロード確認】
モデルのロード成功！
ModernBertForMaskedLM(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50000, 768, padding_idx=0)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=6144, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_fe

README.md:   0%|          | 0.00/26.7k [00:00<?, ?B/s]

train-00000-of-00002.parquet:   0%|          | 0.00/144M [00:00<?, ?B/s]

train-00001-of-00002.parquet:   0%|          | 0.00/147M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/16.7M [00:00<?, ?B/s]

...

データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== microsoft/codebert-base - go Code Search Evaluation Results ====
MRR:         0.0160
MAP:         0.0160
R-Precision: 0.0020

K     Recall    Precision NDCG      F1        Success Rate   Query Cov.     
----------------------------------------------------------------------------
1     0.0020    0.0020    0.0020    0.0020    0.0020         0.0020         
5     0.0160    0.0075    0.0096    0.0098    0.0160         0.0160         
10    0.0380    0.0103    0.0166    0.0147    0.0380         0.0380         
50    0.0980    0.0129    0.0295    0.0196    0.0980         0.0980         
100   0.1590    0.0138    0.0394    0.0214    0.1590         0.1590         

microsoft/graphcodebert-base を評価します (候補プールサイズ: 100)...
データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== microsoft/graphcodebert-base - go Code Search Evaluation Results ====
MRR:         0.2250
MAP:         0.2250
R-Precision: 0.1550

Some weights of the model checkpoint at huggingface/CodeBERTa-small-v1 were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== huggingface/CodeBERTa-small-v1 - go Code Search Evaluation Results ====
MRR:         0.3832
MAP:         0.3832
R-Precision: 0.2750

K     Recall    Precision NDCG      F1        Success Rate   Query Cov.     
----------------------------------------------------------------------------
1     0.2750    0.2750    0.2750    0.2750    0.2750         0.2750         
5     0.4930    0.3560    0.3901    0.3905    0.4930         0.4930         
10    0.6030    0.3708    0.4257    0.4164    0.6030         0.6030         
50    0.8300    0.3816    0.4760    0.4369    0.8300         0.8300         
100   0.9080    0.3827    0.4887    0.4391    0.9080         0.9080         

Shuu12121/CodeMorph-ModernBERT-ALT を評価します (候補プールサイズ: 100)...
データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== Shuu12121/CodeMorph-ModernBERT-ALT - go Code Search Evaluation Results ====
MRR:         0.3812
MAP:         0.3812


You are using a model of type codet5p_bimodal to instantiate a model of type t5. This is not supported for all configurations of models and can yield errors.


データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== Salesforce/codet5p-220m-bimodal - go Code Search Evaluation Results ====
MRR:         0.6201
MAP:         0.6201
R-Precision: 0.5230

K     Recall    Precision NDCG      F1        Success Rate   Query Cov.     
----------------------------------------------------------------------------
1     0.5230    0.5230    0.5230    0.5230    0.5230         0.5230         
5     0.7420    0.6038    0.6382    0.6386    0.7420         0.7420         
10    0.8080    0.6128    0.6597    0.6544    0.8080         0.8080         
50    0.9250    0.6193    0.6870    0.6666    0.9250         0.9250         
100   0.9600    0.6198    0.6927    0.6676    0.9600         0.9600         

Shuu12121/CodeModernBERT-Snake を評価します (候補プールサイズ: 100)...
データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== Shuu12121/CodeModernBERT-Snake - go Code Search Evaluation Results ====
MRR:         0.6409
MAP:         0.6409
R-Preci

In [ ]:
import torch
import numpy as np
import random
import re
import sys
import importlib.util
from datasets import load_dataset
from transformers import AutoModelForMaskedLM, AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity

# sentence_transformers がインストールされているか確認
st_available = importlib.util.find_spec("sentence_transformers") is not None
if not st_available:
    print("sentence_transformers がインストールされていません。一部のモデルでエラーが発生する可能性があります。")
    print("インストールするには: pip install sentence-transformers>=2.7.0")
    print("インストールなしで続行します...")
else:
    print("sentence_transformers が利用可能です。")

# ---------------------------------------------------
# ユーティリティ関数群
# ---------------------------------------------------

def remove_multiline_strings(code):
    """
    Pythonコード中のマルチライン文字列（""" """ や ''' '''）を削除します。
    """
    pattern = r'("""(.*?)"""|\'\'\'(.*?)\'\'\')'
    return re.sub(pattern, '', code, flags=re.DOTALL)

def get_cls_embedding(model, tokenizer, text, device, max_length=256):
    """
    入力テキストの埋め込みを取得する関数。
    - BERT/CodeBERT系の場合は CLS トークンの埋め込みを使用。
    - CodeT5系の場合は encoder 出力の平均プーリングを使用。
    - ModernBERT のように内部に model がある場合は model.model を利用。
    - SFR-Embedding-Code モデルには特別な処理を適用。
    """
    model_name = getattr(model, "name_or_path", "")
    is_sfr_model = "SFR-Embedding-Code" in model_name

    # SFR-Embedding-Code モデルの場合、sentence-transformers 互換の呼び出し方法を使用
    if is_sfr_model:
        encoded_input = tokenizer.encode(text, max_length=max_length, truncation=True, return_tensors='pt').to(device)
        with torch.no_grad():
            try:
                # SFR-Embedding-Code モデルでは encode_tokens メソッドを使う
                if hasattr(model, "encode_tokens"):
                    embedding = model.encode_tokens(encoded_input)
                # または forward_features メソッドを使う
                elif hasattr(model, "forward_features"):
                    embedding = model.forward_features(encoded_input)
                # または last_hidden_state を直接取得
                else:
                    outputs = model(encoded_input)
                    embedding = outputs.last_hidden_state[:, 0, :]
                return embedding.detach().cpu().numpy()
            except Exception as e:
                print(f"SFR モデル特有の処理中にエラー発生: {e}")
                # フォールバックとして、encode メソッドを試す
                try:
                    embedding = model.encode([text], convert_to_tensor=True)
                    return embedding.detach().cpu().numpy()
                except Exception as e2:
                    print(f"フォールバック処理中にもエラー発生: {e2}")
                    raise

    # 通常のモデル処理
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
    # ModernBERT の場合、token_type_ids が不要なので削除する
    if "token_type_ids" in inputs:
        inputs.pop("token_type_ids")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        if hasattr(model, "model"):
            outputs = model.model(**inputs)
            embedding = outputs.last_hidden_state[:, 0, :]
        elif hasattr(model, "bert"):
            outputs = model.bert(**inputs)
            embedding = outputs.last_hidden_state[:, 0, :]
        elif hasattr(model, "roberta"):
            outputs = model.roberta(**inputs)
            embedding = outputs.last_hidden_state[:, 0, :]
        elif hasattr(model, "encoder"):
            # CodeT5系の場合は、CLSトークンが存在しないため平均プーリングを使用
            outputs = model.encoder(**inputs)
            embedding = outputs.last_hidden_state.mean(dim=1)
        else:
            try:
                outputs = model(**inputs)
                if hasattr(outputs, "last_hidden_state"):
                    embedding = outputs.last_hidden_state[:, 0, :]
                elif hasattr(outputs, "hidden_states"):
                    embedding = outputs.hidden_states[-1][:, 0, :]
                else:
                    raise ValueError("モデル出力にlast_hidden_stateまたはhidden_statesがありません。")
            except Exception as e:
                print(f"一般的なモデル処理でエラー: {e}")
                # 特殊なモデルの場合は、forward メソッドに直接 input_ids を渡してみる
                try:
                    outputs = model(inputs["input_ids"])
                    embedding = outputs.last_hidden_state[:, 0, :]
                except Exception as e2:
                    print(f"代替処理もエラー: {e2}")
                    raise

    return embedding.detach().cpu().numpy()


def calculate_f1(precision, recall):
    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)

def calculate_dcg(ranking):
    dcg = 0.0
    for i, rel in enumerate(ranking):
        dcg += rel / np.log2(i + 2)
    return dcg

def calculate_metrics(sim_matrix, k_values=[1, 5, 10, 50, 100]):
    """
    候補プール内で正解（常にインデックス0が正解と仮定）に基づいて評価指標を計算します。
    """
    num_queries = sim_matrix.shape[0]
    metrics = {
        "mrr": 0.0,
        "map": 0.0,
        "r_precision": 0.0,
        "recall@k": {k: 0.0 for k in k_values},
        "precision@k": {k: 0.0 for k in k_values},
        "ndcg@k": {k: 0.0 for k in k_values},
        "f1@k": {k: 0.0 for k in k_values},
        "success_rate@k": {k: 0.0 for k in k_values},
        "query_coverage@k": {k: 0.0 for k in k_values},
    }

    for i in range(num_queries):
        sims = sim_matrix[i]
        ranked_indices = np.argsort(-sims)
        # 正解は候補プール内の先頭（インデックス0）
        correct_rank = np.where(ranked_indices == 0)[0][0] + 1
        metrics["mrr"] += 1.0 / correct_rank
        metrics["map"] += 1.0 / correct_rank
        metrics["r_precision"] += 1.0 if correct_rank == 1 else 0.0

        for k in k_values:
            if correct_rank <= k:
                prec = 1.0 / correct_rank
                rec = 1.0
            else:
                prec = 0.0
                rec = 0.0
            f1 = calculate_f1(prec, rec)
            metrics["precision@k"][k] += prec
            metrics["recall@k"][k] += rec
            metrics["f1@k"][k] += f1

            ideal_ranking = [1.0] + [0.0] * (k - 1)
            actual_ranking = [1.0 if j == 0 else 0.0 for j in ranked_indices[:k]]
            idcg = calculate_dcg(ideal_ranking)
            dcg = calculate_dcg(actual_ranking)
            metrics["ndcg@k"][k] += dcg / idcg if idcg > 0 else 0.0

            metrics["success_rate@k"][k] += 1.0 if correct_rank <= k else 0.0
            metrics["query_coverage@k"][k] += 1.0 if correct_rank <= k else 0.0

    metrics["mrr"] /= num_queries
    metrics["map"] /= num_queries
    metrics["r_precision"] /= num_queries
    for k in k_values:
        metrics["precision@k"][k] /= num_queries
        metrics["recall@k"][k] /= num_queries
        metrics["ndcg@k"][k] /= num_queries
        metrics["f1@k"][k] /= num_queries
        metrics["success_rate@k"][k] /= num_queries
        metrics["query_coverage@k"][k] /= num_queries
    return metrics

def display_code_search_results(metrics, model_name):
    """
    評価結果を見やすいテーブル形式で表示します。
    """
    print(f"\n==== {model_name} Code Search Evaluation Results ====")
    print(f"MRR:         {metrics['mrr']:.4f}")
    print(f"MAP:         {metrics['map']:.4f}")
    print(f"R-Precision: {metrics['r_precision']:.4f}\n")

    header = "{:<6}{:<10}{:<10}{:<10}{:<10}{:<15}{:<15}"
    print(header.format("K", "Recall", "Precision", "NDCG", "F1", "Success Rate", "Query Cov."))
    print("-" * 76)
    for k in sorted(metrics["recall@k"].keys()):
        recall = metrics["recall@k"][k]
        precision = metrics["precision@k"][k]
        ndcg = metrics["ndcg@k"][k]
        f1 = metrics["f1@k"][k]
        success_rate = metrics["success_rate@k"][k]
        query_cov = metrics["query_coverage@k"][k]
        print(header.format(k,
                            f"{recall:.4f}",
                            f"{precision:.4f}",
                            f"{ndcg:.4f}",
                            f"{f1:.4f}",
                            f"{success_rate:.4f}",
                            f"{query_cov:.4f}"))

def evaluate_code_search(model, tokenizer, dataset, device, max_examples=100, pool_size=100,
                         query_field="docstring", code_field="code"):
    """
    指定したデータセット上でコード検索タスクを実施し、評価指標を計算します。
    各サンプルに対して、正解は候補プール内の先頭（自身）と仮定しています。
    """
    num_examples = min(len(dataset), max_examples)
    all_codes = [remove_multiline_strings(dataset[i][code_field]) for i in range(num_examples)]
    queries = [dataset[i][query_field] for i in range(num_examples)]

    # 各コードの埋め込み計算
    all_code_embeddings = []
    print("データセット全体のコード埋め込みを計算中...")

    # バッチ処理を実装して効率化
    batch_size = 16
    for i in range(0, num_examples, batch_size):
        batch_codes = all_codes[i:min(i+batch_size, num_examples)]
        batch_embeddings = []
        for code in batch_codes:
            emb = get_cls_embedding(model, tokenizer, code, device)
            batch_embeddings.append(emb)
        all_code_embeddings.extend(batch_embeddings)

    all_code_embeddings = np.vstack(all_code_embeddings)
    print("コード埋め込み計算完了.")

    # 各クエリに対し候補プール内の類似度計算
    sim_matrices = []
    print("候補コードプールを作成し、類似度計算中...")
    for i in range(num_examples):
        query_embedding = get_cls_embedding(model, tokenizer, queries[i], device).reshape(1, -1)
        candidate_indices = [i] + random.sample([j for j in range(num_examples) if j != i], pool_size - 1)
        candidate_embeddings = all_code_embeddings[candidate_indices]
        sims = cosine_similarity(query_embedding, candidate_embeddings)[0]
        sim_matrices.append(sims)

    sim_matrix = np.array(sim_matrices)
    print("類似度計算完了.")

    metrics = calculate_metrics(sim_matrix)
    return metrics

# ---------------------------------------------------
# メイン処理：各モデル・各言語でコード検索実験を実施
# ---------------------------------------------------

if __name__ == "__main__":
    # Python の乱数シードを設定（候補サンプルの抽出などに影響します）
    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"使用デバイス: {device}")

    # ------------------------------
    # ① ModernBERT 単体ロード確認
    # ------------------------------
    repo_name = "Shuu12121/CodeMorph-ModernBERT"
    try:
        model_demo = AutoModelForMaskedLM.from_pretrained(repo_name)
        tokenizer_demo = AutoTokenizer.from_pretrained(repo_name)
        model_demo.to(device)
        print("\n【ModernBERT 単体ロード確認】")
        print("モデルのロード成功！")
        print(model_demo)
    except Exception as e:
        print(f"\n【ModernBERT 単体ロード確認】")
        print(f"モデルのロード失敗: {e}")

    # ------------------------------
    # ② 評価データセットの言語一覧（例）
    # ------------------------------
    languages = ["python"]
    max_examples = 1000  # メモリと時間の制約のため、各言語ごとに100サンプルに制限

    # ------------------------------
    # ③ 各モデルでコード検索実験を実施
    # ------------------------------
    model_configs = [
      {"name": "microsoft/codebert-base", "class": AutoModelForMaskedLM},
      {"name": "microsoft/graphcodebert-base", "class": AutoModelForMaskedLM},
      {"name": "huggingface/CodeBERTa-small-v1", "class": AutoModelForMaskedLM},
      {"name": "Shuu12121/CodeMorph-ModernBERT-ALT", "class": AutoModelForMaskedLM},
      {"name": "Shuu12121/CodeMorph-ModernBERT", "class": AutoModelForMaskedLM},
      {"name": "Shuu12121/CodeMorph-ModernBERTv2", "class": AutoModelForMaskedLM},
      {"name": "Shuu12121/CodeMorph-ModernBERT-large-vocab", "class": AutoModelForMaskedLM},
      {"name": "Shuu12121/CodeMorph-ModernBERT-Alternative", "class": AutoModelForMaskedLM},
      {"name": "Salesforce/codet5p-220m-bimodal", "class": T5ForConditionalGeneration},
      {"name": "Shuu12121/CodeModernBERT-Snake", "class": AutoModelForMaskedLM, "use_sentence_transformers": False},
      {"name": "Shuu12121/CodeModernBERT-Owl", "class": AutoModelForMaskedLM, "use_sentence_transformers": False},
      {"name": "Salesforce/SFR-Embedding-Code-400M_R", "class": AutoModel, "use_sentence_transformers": True},
      {"name": "Shuu12121/CodeSearch-ModernBERT-Snake", "class": AutoModelForMaskedLM, "use_sentence_transformers": True},
      {"name": "Shuu12121/CodeSearch-ModernBERT-Owl", "class": AutoModelForMaskedLM, "use_sentence_transformers": True},
  ]

    for lang in languages:
        print(f"\n==== 言語: {lang} の評価を開始 ====")
        try:
            # データセットをロード後、シャッフルしてランダムサンプルを抽出
            dataset = load_dataset("google/code_x_glue_tc_nl_code_search_adv", split="test", trust_remote_code=True)
            dataset = dataset.shuffle(seed=42)
            subset = dataset.select(range(min(max_examples, len(dataset))))

            for config in model_configs:
                model_name = config["name"]
                model_class = config["class"]
                print(f"\n{model_name} を評価します (候補プールサイズ: 100)...")
                try:
                    use_st = config.get("use_sentence_transformers", False)

                    if use_st:
                        try:
                            # sentence-transformers を使用する場合
                            from sentence_transformers import SentenceTransformer
                            print("SentenceTransformer を使用してモデルをロードします...")
                            model = SentenceTransformer(model_name, trust_remote_code=True)
                            model.to(device)
                            tokenizer = model.tokenizer

                            # get_cls_embedding 関数を使用せず、直接 encode メソッドを使用するための関数を定義
                            def st_get_embedding(text, device):
                                with torch.no_grad():
                                    embeddings = model.encode([text], convert_to_tensor=True)
                                    return embeddings.cpu().numpy()

                            # 元の関数を一時的に保存
                            original_get_cls_embedding = get_cls_embedding
                            # 関数をオーバーライド
                            get_cls_embedding = lambda model, tokenizer, text, device, max_length=256: st_get_embedding(text, device)

                        except (ImportError, Exception) as e:
                            print(f"SentenceTransformer のロードに失敗しました: {e}")
                            print("通常の方法でモデルをロードします...")
                            tokenizer = AutoTokenizer.from_pretrained(model_name)
                            model = model_class.from_pretrained(model_name, trust_remote_code=True)
                    else:
                        # 通常の方法でモデルをロード
                        tokenizer = AutoTokenizer.from_pretrained(model_name)
                        model = model_class.from_pretrained(model_name, trust_remote_code=True)

                    model.to(device)
                    model.eval()  # 評価モードに設定

                    metrics = evaluate_code_search(model, tokenizer, subset, device,
                                               max_examples=len(subset),
                                               pool_size=1000,
                                               query_field="docstring",
                                               code_field="code")
                    display_code_search_results(metrics, f"{model_name} - {lang}")

                    # 元の get_cls_embedding 関数を復元（オーバーライドした場合）
                    if use_st and 'original_get_cls_embedding' in locals():
                        get_cls_embedding = original_get_cls_embedding
                        del original_get_cls_embedding

                    # メモリ解放
                    del model
                    del tokenizer
                    torch.cuda.empty_cache()

                except Exception as e:
                    print(f"{model_name} の評価中にエラーが発生しました: {e}")
        except Exception as e:
            print(f"{lang} のデータセットロード中にエラーが発生しました: {e}")

sentence_transformers が利用可能です。
使用デバイス: cuda

【ModernBERT 単体ロード確認】
モデルのロード成功！
ModernBertForMaskedLM(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50000, 768, padding_idx=0)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=6144, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_fe

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

train-00000-of-00002.parquet:   0%|          | 0.00/144M [00:00<?, ?B/s]

train-00001-of-00002.parquet:   0%|          | 0.00/147M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/8.59M [00:00<?, ?B/s]

...

データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== microsoft/codebert-base - python Code Search Evaluation Results ====
MRR:         0.0392
MAP:         0.0392
R-Precision: 0.0210

K     Recall    Precision NDCG      F1        Success Rate   Query Cov.     
----------------------------------------------------------------------------
1     0.0210    0.0210    0.0210    0.0210    0.0210         0.0210         
5     0.0460    0.0293    0.0334    0.0331    0.0460         0.0460         
10    0.0710    0.0328    0.0416    0.0392    0.0710         0.0710         
50    0.1460    0.0358    0.0575    0.0451    0.1460         0.1460         
100   0.2200    0.0368    0.0693    0.0471    0.2200         0.2200         

microsoft/graphcodebert-base を評価します (候補プールサイズ: 100)...
データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== microsoft/graphcodebert-base - python Code Search Evaluation Results ====
MRR:         0.1692
MAP:         0.1692
R-Precision

Some weights of the model checkpoint at huggingface/CodeBERTa-small-v1 were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== huggingface/CodeBERTa-small-v1 - python Code Search Evaluation Results ====
MRR:         0.2575
MAP:         0.2575
R-Precision: 0.1720

K     Recall    Precision NDCG      F1        Success Rate   Query Cov.     
----------------------------------------------------------------------------
1     0.1720    0.1720    0.1720    0.1720    0.1720         0.1720         
5     0.3420    0.2333    0.2603    0.2602    0.3420         0.3420         
10    0.4310    0.2450    0.2889    0.2808    0.4310         0.4310         
50    0.6500    0.2548    0.3365    0.2995    0.6500         0.6500         
100   0.7560    0.2564    0.3538    0.3025    0.7560         0.7560         

Shuu12121/CodeMorph-ModernBERT-ALT を評価します (候補プールサイズ: 100)...
データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== Shuu12121/CodeMorph-ModernBERT-ALT - python Code Search Evaluation Results ====
MRR:         0.4030
MAP:        

You are using a model of type codet5p_bimodal to instantiate a model of type t5. This is not supported for all configurations of models and can yield errors.


データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== Salesforce/codet5p-220m-bimodal - python Code Search Evaluation Results ====
MRR:         0.3040
MAP:         0.3040
R-Precision: 0.2170

K     Recall    Precision NDCG      F1        Success Rate   Query Cov.     
----------------------------------------------------------------------------
1     0.2170    0.2170    0.2170    0.2170    0.2170         0.2170         
5     0.3920    0.2811    0.3086    0.3087    0.3920         0.3920         
10    0.4720    0.2916    0.3343    0.3272    0.4720         0.4720         
50    0.6910    0.3019    0.3826    0.3468    0.6910         0.6910         
100   0.7670    0.3030    0.3950    0.3490    0.7670         0.7670         

Shuu12121/CodeModernBERT-Snake を評価します (候補プールサイズ: 100)...
データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== Shuu12121/CodeModernBERT-Snake - python Code Search Evaluation Results ====
MRR:         0.1577
MAP:         0.1577

In [ ]:
import torch
import numpy as np
import random
import re
import sys
import importlib.util
from datasets import load_dataset
from transformers import AutoModelForMaskedLM, AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity

# sentence_transformers がインストールされているか確認
st_available = importlib.util.find_spec("sentence_transformers") is not None
if not st_available:
    print("sentence_transformers がインストールされていません。一部のモデルでエラーが発生する可能性があります。")
    print("インストールするには: pip install sentence-transformers>=2.7.0")
    print("インストールなしで続行します...")
else:
    print("sentence_transformers が利用可能です。")

# ---------------------------------------------------
# ユーティリティ関数群
# ---------------------------------------------------

def remove_multiline_strings(code):
    """
    Pythonコード中のマルチライン文字列（""" """ や ''' '''）を削除します。
    """
    pattern = r'("""(.*?)"""|\'\'\'(.*?)\'\'\')'
    return re.sub(pattern, '', code, flags=re.DOTALL)

def get_cls_embedding(model, tokenizer, text, device, max_length=256):
    """
    入力テキストの埋め込みを取得する関数。
    - BERT/CodeBERT系の場合は CLS トークンの埋め込みを使用。
    - CodeT5系の場合は encoder 出力の平均プーリングを使用。
    - ModernBERT のように内部に model がある場合は model.model を利用。
    - SFR-Embedding-Code モデルには特別な処理を適用。
    """
    model_name = getattr(model, "name_or_path", "")
    is_sfr_model = "SFR-Embedding-Code" in model_name

    # SFR-Embedding-Code モデルの場合、sentence-transformers 互換の呼び出し方法を使用
    if is_sfr_model:
        encoded_input = tokenizer.encode(text, max_length=max_length, truncation=True, return_tensors='pt').to(device)
        with torch.no_grad():
            try:
                # SFR-Embedding-Code モデルでは encode_tokens メソッドを使う
                if hasattr(model, "encode_tokens"):
                    embedding = model.encode_tokens(encoded_input)
                # または forward_features メソッドを使う
                elif hasattr(model, "forward_features"):
                    embedding = model.forward_features(encoded_input)
                # または last_hidden_state を直接取得
                else:
                    outputs = model(encoded_input)
                    embedding = outputs.last_hidden_state[:, 0, :]
                return embedding.detach().cpu().numpy()
            except Exception as e:
                print(f"SFR モデル特有の処理中にエラー発生: {e}")
                # フォールバックとして、encode メソッドを試す
                try:
                    embedding = model.encode([text], convert_to_tensor=True)
                    return embedding.detach().cpu().numpy()
                except Exception as e2:
                    print(f"フォールバック処理中にもエラー発生: {e2}")
                    raise

    # 通常のモデル処理
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
    # ModernBERT の場合、token_type_ids が不要なので削除する
    if "token_type_ids" in inputs:
        inputs.pop("token_type_ids")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        if hasattr(model, "model"):
            outputs = model.model(**inputs)
            embedding = outputs.last_hidden_state[:, 0, :]
        elif hasattr(model, "bert"):
            outputs = model.bert(**inputs)
            embedding = outputs.last_hidden_state[:, 0, :]
        elif hasattr(model, "roberta"):
            outputs = model.roberta(**inputs)
            embedding = outputs.last_hidden_state[:, 0, :]
        elif hasattr(model, "encoder"):
            # CodeT5系の場合は、CLSトークンが存在しないため平均プーリングを使用
            outputs = model.encoder(**inputs)
            embedding = outputs.last_hidden_state.mean(dim=1)
        else:
            try:
                outputs = model(**inputs)
                if hasattr(outputs, "last_hidden_state"):
                    embedding = outputs.last_hidden_state[:, 0, :]
                elif hasattr(outputs, "hidden_states"):
                    embedding = outputs.hidden_states[-1][:, 0, :]
                else:
                    raise ValueError("モデル出力にlast_hidden_stateまたはhidden_statesがありません。")
            except Exception as e:
                print(f"一般的なモデル処理でエラー: {e}")
                # 特殊なモデルの場合は、forward メソッドに直接 input_ids を渡してみる
                try:
                    outputs = model(inputs["input_ids"])
                    embedding = outputs.last_hidden_state[:, 0, :]
                except Exception as e2:
                    print(f"代替処理もエラー: {e2}")
                    raise

    return embedding.detach().cpu().numpy()


def calculate_f1(precision, recall):
    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)

def calculate_dcg(ranking):
    dcg = 0.0
    for i, rel in enumerate(ranking):
        dcg += rel / np.log2(i + 2)
    return dcg

def calculate_metrics(sim_matrix, k_values=[1, 5, 10, 50, 100]):
    """
    候補プール内で正解（常にインデックス0が正解と仮定）に基づいて評価指標を計算します。
    """
    num_queries = sim_matrix.shape[0]
    metrics = {
        "mrr": 0.0,
        "map": 0.0,
        "r_precision": 0.0,
        "recall@k": {k: 0.0 for k in k_values},
        "precision@k": {k: 0.0 for k in k_values},
        "ndcg@k": {k: 0.0 for k in k_values},
        "f1@k": {k: 0.0 for k in k_values},
        "success_rate@k": {k: 0.0 for k in k_values},
        "query_coverage@k": {k: 0.0 for k in k_values},
    }

    for i in range(num_queries):
        sims = sim_matrix[i]
        ranked_indices = np.argsort(-sims)
        # 正解は候補プール内の先頭（インデックス0）
        correct_rank = np.where(ranked_indices == 0)[0][0] + 1
        metrics["mrr"] += 1.0 / correct_rank
        metrics["map"] += 1.0 / correct_rank
        metrics["r_precision"] += 1.0 if correct_rank == 1 else 0.0

        for k in k_values:
            if correct_rank <= k:
                prec = 1.0 / correct_rank
                rec = 1.0
            else:
                prec = 0.0
                rec = 0.0
            f1 = calculate_f1(prec, rec)
            metrics["precision@k"][k] += prec
            metrics["recall@k"][k] += rec
            metrics["f1@k"][k] += f1

            ideal_ranking = [1.0] + [0.0] * (k - 1)
            actual_ranking = [1.0 if j == 0 else 0.0 for j in ranked_indices[:k]]
            idcg = calculate_dcg(ideal_ranking)
            dcg = calculate_dcg(actual_ranking)
            metrics["ndcg@k"][k] += dcg / idcg if idcg > 0 else 0.0

            metrics["success_rate@k"][k] += 1.0 if correct_rank <= k else 0.0
            metrics["query_coverage@k"][k] += 1.0 if correct_rank <= k else 0.0

    metrics["mrr"] /= num_queries
    metrics["map"] /= num_queries
    metrics["r_precision"] /= num_queries
    for k in k_values:
        metrics["precision@k"][k] /= num_queries
        metrics["recall@k"][k] /= num_queries
        metrics["ndcg@k"][k] /= num_queries
        metrics["f1@k"][k] /= num_queries
        metrics["success_rate@k"][k] /= num_queries
        metrics["query_coverage@k"][k] /= num_queries
    return metrics

def display_code_search_results(metrics, model_name):
    """
    評価結果を見やすいテーブル形式で表示します。
    """
    print(f"\n==== {model_name} Code Search Evaluation Results ====")
    print(f"MRR:         {metrics['mrr']:.4f}")
    print(f"MAP:         {metrics['map']:.4f}")
    print(f"R-Precision: {metrics['r_precision']:.4f}\n")

    header = "{:<6}{:<10}{:<10}{:<10}{:<10}{:<15}{:<15}"
    print(header.format("K", "Recall", "Precision", "NDCG", "F1", "Success Rate", "Query Cov."))
    print("-" * 76)
    for k in sorted(metrics["recall@k"].keys()):
        recall = metrics["recall@k"][k]
        precision = metrics["precision@k"][k]
        ndcg = metrics["ndcg@k"][k]
        f1 = metrics["f1@k"][k]
        success_rate = metrics["success_rate@k"][k]
        query_cov = metrics["query_coverage@k"][k]
        print(header.format(k,
                            f"{recall:.4f}",
                            f"{precision:.4f}",
                            f"{ndcg:.4f}",
                            f"{f1:.4f}",
                            f"{success_rate:.4f}",
                            f"{query_cov:.4f}"))

def evaluate_code_search(model, tokenizer, dataset, device, max_examples=100, pool_size=100,
                         query_field="docstring", code_field="code"):
    """
    指定したデータセット上でコード検索タスクを実施し、評価指標を計算します。
    各サンプルに対して、正解は候補プール内の先頭（自身）と仮定しています。
    """
    num_examples = min(len(dataset), max_examples)
    all_codes = [remove_multiline_strings(dataset[i][code_field]) for i in range(num_examples)]
    queries = [dataset[i][query_field] for i in range(num_examples)]

    # 各コードの埋め込み計算
    all_code_embeddings = []
    print("データセット全体のコード埋め込みを計算中...")

    # バッチ処理を実装して効率化
    batch_size = 16
    for i in range(0, num_examples, batch_size):
        batch_codes = all_codes[i:min(i+batch_size, num_examples)]
        batch_embeddings = []
        for code in batch_codes:
            emb = get_cls_embedding(model, tokenizer, code, device)
            batch_embeddings.append(emb)
        all_code_embeddings.extend(batch_embeddings)
    all_code_embeddings = np.vstack(all_code_embeddings)
    print("コード埋め込み計算完了.")

    # 各クエリに対し候補プール内の類似度計算
    sim_matrices = []
    print("候補コードプールを作成し、類似度計算中...")
    for i in range(num_examples):
        query_embedding = get_cls_embedding(model, tokenizer, queries[i], device).reshape(1, -1)
        candidate_indices = [i] + random.sample([j for j in range(num_examples) if j != i], pool_size - 1)
        candidate_embeddings = all_code_embeddings[candidate_indices]
        sims = cosine_similarity(query_embedding, candidate_embeddings)[0]
        sim_matrices.append(sims)

    sim_matrix = np.array(sim_matrices)
    print("類似度計算完了.")

    metrics = calculate_metrics(sim_matrix)
    return metrics

# ---------------------------------------------------
# メイン処理：各モデル・各言語でコード検索実験を実施
# ---------------------------------------------------

if __name__ == "__main__":
    # Python の乱数シードを設定（候補サンプルの抽出などに影響します）
    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"使用デバイス: {device}")

    # ------------------------------
    # ① ModernBERT 単体ロード確認
    # ------------------------------
    repo_name = "Shuu12121/CodeMorph-ModernBERT"
    try:
        model_demo = AutoModelForMaskedLM.from_pretrained(repo_name)
        tokenizer_demo = AutoTokenizer.from_pretrained(repo_name)
        model_demo.to(device)
        print("\n【ModernBERT 単体ロード確認】")
        print("モデルのロード成功！")
        print(model_demo)
    except Exception as e:
        print(f"\n【ModernBERT 単体ロード確認】")
        print(f"モデルのロード失敗: {e}")

    # ------------------------------
    # ② 評価データセットの言語一覧（例）
    # ------------------------------
    languages = ["python", "java", "javascript", "php", "ruby", "go"]
    max_examples = 1000  # メモリと時間の制約のため、各言語ごとに100サンプルに制限

    # ------------------------------
    # ③ 各モデルでコード検索実験を実施
    # ------------------------------
    model_configs = [
      {"name": "microsoft/codebert-base", "class": AutoModelForMaskedLM},
      {"name": "microsoft/graphcodebert-base", "class": AutoModelForMaskedLM},
      {"name": "huggingface/CodeBERTa-small-v1", "class": AutoModelForMaskedLM},
      {"name": "Shuu12121/CodeMorph-ModernBERT-ALT", "class": AutoModelForMaskedLM},
      {"name": "Shuu12121/CodeMorph-ModernBERT", "class": AutoModelForMaskedLM},
      {"name": "Shuu12121/CodeMorph-ModernBERTv2", "class": AutoModelForMaskedLM},
      {"name": "Shuu12121/CodeMorph-ModernBERT-large-vocab", "class": AutoModelForMaskedLM},
      {"name": "Shuu12121/CodeMorph-ModernBERT-Alternative", "class": AutoModelForMaskedLM},
      {"name": "Salesforce/codet5p-220m-bimodal", "class": T5ForConditionalGeneration},
      {"name": "Shuu12121/CodeModernBERT-Snake", "class": AutoModelForMaskedLM, "use_sentence_transformers": False},
      {"name": "Shuu12121/CodeModernBERT-Owl", "class": AutoModelForMaskedLM, "use_sentence_transformers": False},
      {"name": "Salesforce/SFR-Embedding-Code-400M_R", "class": AutoModel, "use_sentence_transformers": True},
      {"name": "Shuu12121/CodeSearch-ModernBERT-Snake", "class": AutoModelForMaskedLM, "use_sentence_transformers": True},
      {"name": "Shuu12121/CodeSearch-ModernBERT-Owl", "class": AutoModelForMaskedLM, "use_sentence_transformers": True},
    ]

    for lang in languages:
        print(f"\n==== 言語: {lang} の評価を開始 ====")
        try:
            # データセットをロード後、シャッフルしてランダムサンプルを抽出
            dataset = load_dataset("code_search_net", lang, split="test", trust_remote_code=True)
            dataset = dataset.shuffle(seed=42)
            subset = dataset.select(range(min(max_examples, len(dataset))))

            for config in model_configs:
                model_name = config["name"]
                model_class = config["class"]
                print(f"\n{model_name} を評価します (候補プールサイズ: 100)...")
                try:
                    use_st = config.get("use_sentence_transformers", False)

                    if use_st:
                        try:
                            # sentence-transformers を使用する場合
                            from sentence_transformers import SentenceTransformer
                            print("SentenceTransformer を使用してモデルをロードします...")
                            model = SentenceTransformer(model_name, trust_remote_code=True)
                            model.to(device)
                            tokenizer = model.tokenizer

                            # get_cls_embedding 関数を使用せず、直接 encode メソッドを使用するための関数を定義
                            def st_get_embedding(text, device):
                                with torch.no_grad():
                                    embeddings = model.encode([text], convert_to_tensor=True)
                                    return embeddings.cpu().numpy()

                            # 元の関数を一時的に保存
                            original_get_cls_embedding = get_cls_embedding
                            # 関数をオーバーライド
                            get_cls_embedding = lambda model, tokenizer, text, device, max_length=256: st_get_embedding(text, device)

                        except (ImportError, Exception) as e:
                            print(f"SentenceTransformer のロードに失敗しました: {e}")
                            print("通常の方法でモデルをロードします...")
                            tokenizer = AutoTokenizer.from_pretrained(model_name)
                            model = model_class.from_pretrained(model_name, trust_remote_code=True)
                    else:
                        # 通常の方法でモデルをロード
                        tokenizer = AutoTokenizer.from_pretrained(model_name)
                        model = model_class.from_pretrained(model_name, trust_remote_code=True)

                    model.to(device)
                    model.eval()  # 評価モードに設定

                    metrics = evaluate_code_search(model, tokenizer, subset, device,
                                               max_examples=len(subset),
                                               pool_size=1000,
                                               query_field="func_documentation_string",
                                               code_field="func_code_string")
                    display_code_search_results(metrics, f"{model_name} - {lang}")

                    # 元の get_cls_embedding 関数を復元（オーバーライドした場合）
                    if use_st and 'original_get_cls_embedding' in locals():
                        get_cls_embedding = original_get_cls_embedding
                        del original_get_cls_embedding

                    # メモリ解放
                    del model
                    del tokenizer
                    torch.cuda.empty_cache()

                except Exception as e:
                    print(f"{model_name} の評価中にエラーが発生しました: {e}")
        except Exception as e:
            print(f"{lang} のデータセットロード中にエラーが発生しました: {e}")

sentence_transformers が利用可能です。
使用デバイス: cuda

【ModernBERT 単体ロード確認】
モデルのロード成功！
ModernBertForMaskedLM(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50000, 768, padding_idx=0)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=6144, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_fe

README.md:   0%|          | 0.00/12.9k [00:00<?, ?B/s]

code_search_net.py:   0%|          | 0.00/8.44k [00:00<?, ?B/s]

python.zip:   0%|          | 0.00/941M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/412178 [00:00<?, ? examples/s]

...

データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== microsoft/codebert-base - go Code Search Evaluation Results ====
MRR:         0.0247
MAP:         0.0247
R-Precision: 0.0100

K     Recall    Precision NDCG      F1        Success Rate   Query Cov.     
----------------------------------------------------------------------------
1     0.0100    0.0100    0.0100    0.0100    0.0100         0.0100         
5     0.0310    0.0167    0.0202    0.0199    0.0310         0.0310         
10    0.0390    0.0177    0.0227    0.0217    0.0390         0.0390         
50    0.1280    0.0215    0.0418    0.0289    0.1280         0.1280         
100   0.1850    0.0223    0.0510    0.0305    0.1850         0.1850         

microsoft/graphcodebert-base を評価します (候補プールサイズ: 100)...
データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== microsoft/graphcodebert-base - go Code Search Evaluation Results ====
MRR:         0.2162
MAP:         0.2162
R-Precision: 0.1420

Some weights of the model checkpoint at huggingface/CodeBERTa-small-v1 were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== huggingface/CodeBERTa-small-v1 - go Code Search Evaluation Results ====
MRR:         0.3213
MAP:         0.3213
R-Precision: 0.2280

K     Recall    Precision NDCG      F1        Success Rate   Query Cov.     
----------------------------------------------------------------------------
1     0.2280    0.2280    0.2280    0.2280    0.2280         0.2280         
5     0.4040    0.2930    0.3206    0.3209    0.4040         0.4040         
10    0.5090    0.3070    0.3545    0.3455    0.5090         0.5090         
50    0.7720    0.3191    0.4121    0.3685    0.7720         0.7720         
100   0.8800    0.3207    0.4298    0.3717    0.8800         0.8800         

Shuu12121/CodeMorph-ModernBERT-ALT を評価します (候補プールサイズ: 100)...
データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== Shuu12121/CodeMorph-ModernBERT-ALT - go Code Search Evaluation Results ====
MRR:         0.3121
MAP:         0.3121


You are using a model of type codet5p_bimodal to instantiate a model of type t5. This is not supported for all configurations of models and can yield errors.


データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== Salesforce/codet5p-220m-bimodal - go Code Search Evaluation Results ====
MRR:         0.5080
MAP:         0.5080
R-Precision: 0.4180

K     Recall    Precision NDCG      F1        Success Rate   Query Cov.     
----------------------------------------------------------------------------
1     0.4180    0.4180    0.4180    0.4180    0.4180         0.4180         
5     0.6040    0.4872    0.5164    0.5170    0.6040         0.6040         
10    0.6790    0.4974    0.5408    0.5349    0.6790         0.6790         
50    0.8740    0.5067    0.5839    0.5524    0.8740         0.8740         
100   0.9380    0.5077    0.5944    0.5543    0.9380         0.9380         

Shuu12121/CodeModernBERT-Snake を評価します (候補プールサイズ: 100)...
データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== Shuu12121/CodeModernBERT-Snake - go Code Search Evaluation Results ====
MRR:         0.5230
MAP:         0.5230
R-Preci

In [ ]:

# prompt: 切断する

from google.colab import runtime
runtime.unassign()